# 01 — Preprocessing

Load TCGA RNA-seq and GEO microarray expression data, harmonize gene identifiers,
apply ComBat batch correction across the three breast GEO cohorts, and run ssGSEA
pathway scoring against the MSigDB v2023.2 collections.

**Outputs:**
- `geo_clinical.pkl`, `tcga_clinical.pkl` — survival annotation per cohort
- `geo_expr_corrected.pkl`, `tcga_expr_corrected.pkl` — normalized expression matrices
- `ssgsea_corrected.pkl` — pathway enrichment scores
- `ssgsea_raw.pkl` — uncorrected pathway scores (batch sensitivity check)
- `combined_genesets.pkl` — full gene set dictionary

**Endpoints used:** DFI as primary recurrence endpoint, PFI as fallback. GBM uses
PFI as primary because DFI is not meaningful for an infiltrative primary brain tumor.


In [ ]:
import gzip
import math
import os
import pickle
import random
import tempfile
import warnings
from pathlib import Path

import gseapy as gp
import numpy as np
import pandas as pd
from sklearn.preprocessing import quantile_transform

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


## Paths

Set `BASE_DIR` to the root of your data directory. Expected layout:

```
BASE_DIR/
  Raw Data/
    geo/{GSE2034,GSE2990,GSE103746,GSE31210}/
    tcga_rnaseq/{TCGA-BRCA,TCGA-LUAD,TCGA-KIRC,TCGA-GBM}/
    tcga_clinical/TCGA-CDR.csv
    pathway_databases/{h.all.gmt,c2.cp.kegg_legacy.gmt,c2.cp.reactome.gmt,c5.go.bp.gmt}
  intermediates/
```


In [ ]:
BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
RAW_DIR = BASE_DIR / "Raw Data"
INTER_DIR = BASE_DIR / "intermediates"
INTER_DIR.mkdir(parents=True, exist_ok=True)

DAYS_PER_MONTH = 365.25 / 12


## Cohort registry

Expected sample sizes and time units, used for validation during loading.


In [ ]:
EXPECTED_N = {
    "GSE2034": 286,
    "GSE2990": 189,
    "GSE103746": 508,
    "GSE31210": 226,
    "TCGA-BRCA": 1098,
    "TCGA-LUAD": 585,
    "TCGA-KIRC": 530,
    "TCGA-GBM": 154,
}

FLOOR_N = {
    "GSE2034": 270, "GSE2990": 170, "GSE103746": 160, "GSE31210": 210,
    "TCGA-BRCA": 900, "TCGA-LUAD": 280, "TCGA-KIRC": 100, "TCGA-GBM": 50,
}

TIME_UNITS = {
    "GSE2034": "months", "GSE2990": "months",
    "GSE103746": "months", "GSE31210": "days",
    "TCGA-BRCA": "days", "TCGA-LUAD": "days",
    "TCGA-KIRC": "days", "TCGA-GBM": "days",
}

# GBM uses PFI as primary because DFI is undefined for a primary brain tumor
# that is progressive from diagnosis. All other TCGA cancers prefer DFI.
CANCER_ENDPOINT_PRIORITY = {
    "TCGA-GBM": ["PFI", "OS", "DSS"],
    "TCGA-BRCA": ["DFI", "PFI"],
    "TCGA-LUAD": ["DFI", "PFI"],
    "TCGA-KIRC": ["DFI", "PFI"],
}
DEFAULT_ENDPOINT_PRIORITY = ["DFI", "PFI"]


## Utility functions


In [ ]:
NON_TIME_LABELS = frozenset([
    "age", "birth", "diagnosis", "grade", "stage",
    "treatment", "surgery", "chemotherapy", "histol",
])


def detect_time_unit(col_label):
    """Infer time unit from a column header. Returns 'years', 'days',
    'months', or None if no signal."""
    if col_label is None:
        return None
    lab = str(col_label).lower()
    if any(neg in lab for neg in NON_TIME_LABELS):
        return None
    if any(k in lab for k in ("year", " yr", "_yr")):
        return "years"
    if "day" in lab:
        return "days"
    if any(k in lab for k in ("month", " mo", "_mo")):
        return "months"
    return None


def apply_time_units(clinical_df, cohort_id, time_col_label=None):
    """Convert clinical time column to months. Apply a sanity check:
    after conversion, max should be < 600 months and median > 0.5."""
    label_unit = detect_time_unit(time_col_label)
    dict_unit = TIME_UNITS.get(cohort_id, "months")
    unit = label_unit if label_unit is not None else dict_unit
    clinical_df = clinical_df.copy()
    if unit == "days":
        clinical_df["time"] = clinical_df["time"] / DAYS_PER_MONTH
    elif unit == "years":
        clinical_df["time"] = clinical_df["time"] * 12.0
    t = clinical_df["time"].astype(float)
    if t.max() > 600 or t.median() < 0.5:
        raise ValueError(
            f"{cohort_id}: time sanity check failed after conversion. "
            f"range=[{t.min():.2f}, {t.max():.2f}] months, median={t.median():.2f}"
        )
    return clinical_df


def validate_sample_size(cohort_id, n_loaded):
    """Raise if loaded sample size is below the registered floor."""
    floor = FLOOR_N.get(cohort_id)
    if floor is not None and n_loaded < floor:
        raise ValueError(
            f"{cohort_id}: loaded n={n_loaded} below floor n={floor}"
        )
    expected = EXPECTED_N.get(cohort_id)
    mark = "ok" if (expected is None or n_loaded == expected) else "diff"
    print(f"  [{mark}] {cohort_id}: n={n_loaded} (expected {expected})")


## GEO loading

Parse Series Matrix files, build probe-to-gene maps from the SOFT family files,
and collapse probes to genes by selecting the highest-mean probe per gene.


In [ ]:
def parse_series_matrix(filepath):
    """Parse a GEO Series Matrix file. Returns (expression matrix, characteristics)."""
    expr_lines, sample_ids, char_dicts, header = [], None, None, None
    opener = gzip.open if str(filepath).endswith(".gz") else open
    with opener(filepath, "rt", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line.startswith("!Sample_geo_accession"):
                parts = line.split("\t")
                sample_ids = [v.strip('"') for v in parts[1:]]
                char_dicts = [{} for _ in sample_ids]
            elif line.startswith("!Sample_characteristics_ch1"):
                parts = line.split("\t")
                vals = [v.strip('"') for v in parts[1:]]
                for i, v in enumerate(vals):
                    if i < len(char_dicts) and ":" in v:
                        k, val = v.split(":", 1)
                        char_dicts[i].setdefault(k.strip().lower(), val.strip())
            elif line.startswith('"ID_REF"'):
                header = [h.strip('"') for h in line.split("\t")]
            elif header and not line.startswith("!") and line:
                parts = line.split("\t")
                if len(parts) == len(header):
                    expr_lines.append(parts)
    if not expr_lines or sample_ids is None:
        return None, None
    expr_df = pd.DataFrame(expr_lines, columns=header).set_index("ID_REF")
    expr_df.index = expr_df.index.astype(str).str.replace('"', "", regex=False).str.strip()
    expr_df = expr_df.apply(pd.to_numeric, errors="coerce")
    chars_df = pd.DataFrame(char_dicts, index=sample_ids)
    chars_df.index.name = "sample_id"
    return expr_df, chars_df


def build_probe_gene_map(soft_path, platform):
    """Extract probe -> gene symbol map from a SOFT family file."""
    probe_gene = {}
    in_table, headers, gene_col_idx, id_col_idx = False, None, None, 0
    opener = gzip.open if str(soft_path).endswith(".gz") else open
    with opener(soft_path, "rt", errors="replace") as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                in_table = True
                continue
            elif line.startswith("!platform_table_end"):
                break
            if not in_table:
                continue
            parts = line.strip().split("\t")
            if headers is None:
                headers = [h.strip('"').lower() for h in parts]
                if platform == "GPL10558":
                    for col_name in ["symbol", "ilmn_gene"]:
                        if col_name in headers:
                            gene_col_idx = headers.index(col_name)
                            break
                elif "gene symbol" in headers:
                    gene_col_idx = headers.index("gene symbol")
                continue
            if gene_col_idx is None or len(parts) <= gene_col_idx:
                continue
            probe = parts[id_col_idx].strip('"')
            gene = parts[gene_col_idx].strip('"').strip()
            if "///" in gene:
                gene = gene.split("///")[0].strip()
            if gene and gene != "---":
                probe_gene[probe] = gene
    return probe_gene


def collapse_probes_to_genes(expr_df, probe_gene_map):
    """Collapse probe-level expression to gene-level by selecting the
    highest-mean probe per gene symbol."""
    mapped = (
        expr_df.rename(index=probe_gene_map)
        .assign(gene=lambda df: df.index)
        .assign(mean_expr=lambda df: df.drop("gene", axis=1).mean(axis=1))
    )
    mapped = mapped[mapped["gene"].notna() & (mapped["gene"] != "")]
    mapped = mapped[~mapped["gene"].astype(str).str.match(r"^\d+(\.\d+)?$")]
    mapped = mapped.sort_values("mean_expr", ascending=False)
    gene_expr = mapped.drop_duplicates(subset="gene", keep="first")
    gene_expr = gene_expr.set_index("gene").drop(columns=["mean_expr"])
    gene_expr.index = gene_expr.index.astype(str)
    return gene_expr


In [ ]:
GEO_CONFIG = {
    "GSE2034": {
        "matrix": RAW_DIR / "geo" / "GSE2034" / "GSE2034_series_matrix.txt",
        "soft": RAW_DIR / "geo" / "GSE2034" / "GSE2034_family.soft.gz",
        "platform": "GPL96",
    },
    "GSE2990": {
        "matrix": RAW_DIR / "geo" / "GSE2990" / "GSE2990_series_matrix.txt",
        "soft": RAW_DIR / "geo" / "GSE2990" / "GSE2990_family.soft.gz",
        "suppl": RAW_DIR / "geo" / "GSE2990" / "GSE2990_suppl_info.txt",
        "platform": "GPL96",
    },
    "GSE103746": {
        "matrix": RAW_DIR / "geo" / "GSE103746" / "GSE103746-GPL10558_series_matrix.txt",
        "soft": RAW_DIR / "geo" / "GSE103746" / "GSE103746_family.soft.gz",
        "platform": "GPL10558",
    },
    "GSE31210": {
        "matrix": RAW_DIR / "geo" / "GSE31210" / "GSE31210_series_matrix.txt",
        "soft": RAW_DIR / "geo" / "GSE31210" / "GSE31210_family.soft.gz",
        "platform": "GPL570",
    },
}

geo_expr = {}
geo_clinical = {}

for gse_id, cfg in GEO_CONFIG.items():
    print(f"\n{gse_id} ({cfg['platform']})")
    matrix_path = cfg["matrix"] if cfg["matrix"].exists() else Path(str(cfg["matrix"]) + ".gz")
    if not matrix_path.exists():
        print(f"  matrix not found: {cfg['matrix']}")
        continue
    expr, chars_df = parse_series_matrix(str(matrix_path))
    if expr is None:
        continue
    soft_path = cfg["soft"] if cfg["soft"].exists() else Path(str(cfg["soft"]).replace(".gz", ""))
    probe_map = build_probe_gene_map(str(soft_path), cfg["platform"])
    gene_expr = collapse_probes_to_genes(expr, probe_map)
    geo_expr[gse_id] = gene_expr

    clinical = pd.DataFrame(index=chars_df.index)
    time_label = None

    if gse_id == "GSE2034":
        # DMFS clinical comes from a manual sidecar CSV with curated time/event
        manual = RAW_DIR / "geo" / "GSE2034" / "GSE2034_clinical.csv"
        if not manual.exists():
            raise FileNotFoundError(f"{gse_id}: expected manual clinical file {manual}")
        mc = pd.read_csv(manual)
        for id_col in ["sample_id", "geo_accession", "GSM"]:
            if id_col in mc.columns:
                mc = mc.set_index(id_col)
                break
        else:
            mc = mc.set_index(mc.columns[0])
        clinical["time"] = pd.to_numeric(
            mc.get("dmfs_time", mc.get("time", None)), errors="coerce"
        ).reindex(chars_df.index).values
        clinical["event"] = pd.to_numeric(
            mc.get("dmfs_event", mc.get("event", None)), errors="coerce"
        ).reindex(chars_df.index).values
        time_label = "dmfs_time (months)"

    elif gse_id == "GSE2990":
        suppl_path = cfg.get("suppl")
        if suppl_path is None or not Path(suppl_path).exists():
            raise FileNotFoundError(f"{gse_id}: expected supplementary file {suppl_path}")
        suppl = pd.read_csv(cfg["suppl"], sep="\t")
        if suppl.shape[1] < 3:
            suppl = pd.read_csv(cfg["suppl"], sep=",")
        if "geo_accn" in suppl.columns:
            suppl = suppl.set_index("geo_accn")
        clinical["event"] = pd.to_numeric(
            suppl.reindex(chars_df.index)["event.dmfs"], errors="coerce"
        )
        clinical["time"] = pd.to_numeric(
            suppl.reindex(chars_df.index)["time.dmfs"], errors="coerce"
        )
        time_label = "time.dmfs (years)"

    elif gse_id == "GSE103746":
        if "ibtr (ipsilateral breast tumor recurrence)" in chars_df.columns:
            clinical["event"] = pd.to_numeric(
                chars_df["ibtr (ipsilateral breast tumor recurrence)"], errors="coerce"
            )
        elif "lr (local recurrence)" in chars_df.columns:
            clinical["event"] = pd.to_numeric(
                chars_df["lr (local recurrence)"], errors="coerce"
            )
        if "follow-up time (years)" in chars_df.columns:
            clinical["time"] = pd.to_numeric(
                chars_df["follow-up time (years)"], errors="coerce"
            )
            time_label = "follow-up time (years)"

    elif gse_id == "GSE31210":
        if "relapse" in chars_df.columns:
            raw_relapse = chars_df["relapse"].astype(str).str.strip().str.lower()
            clinical["event"] = raw_relapse.map({"relapsed": 1.0, "not relapsed": 0.0})
        if "days before relapse/censor" in chars_df.columns:
            clinical["time"] = pd.to_numeric(
                chars_df["days before relapse/censor"], errors="coerce"
            )
            time_label = "days before relapse/censor"
        elif "months before relapse/censor" in chars_df.columns:
            clinical["time"] = pd.to_numeric(
                chars_df["months before relapse/censor"], errors="coerce"
            )
            time_label = "months before relapse/censor"

    if "event" not in clinical.columns:
        raise KeyError(f"{gse_id}: clinical event column was not parsed")
    if "time" not in clinical.columns:
        raise KeyError(f"{gse_id}: clinical time column was not parsed")
    clinical["event"] = clinical["event"].fillna(0.0)
    clinical = clinical.dropna(subset=["time"]).copy()
    clinical["event"] = pd.to_numeric(clinical["event"], errors="coerce").fillna(0.0)
    clinical["time"] = clinical["time"].astype(float)
    non_binary = ~clinical["event"].isin([0.0, 1.0])
    if non_binary.any():
        clinical["event"] = (clinical["event"] > 0).astype(float)
    clinical = apply_time_units(clinical, gse_id, time_col_label=time_label)
    geo_clinical[gse_id] = clinical
    print(f"  loaded {len(clinical)} patients, {int(clinical['event'].sum())} events")
    validate_sample_size(gse_id, len(clinical))


## TCGA loading

Read STAR-aligned gene counts from per-sample files, harmonize Ensembl IDs to gene
symbols using Gencode v36, and filter to protein-coding genes. Apply TMM normalization
to obtain log2(CPM + 1) values. Endpoint selection follows the per-cancer priority
list defined above.


In [ ]:
def tmm_log2cpm(counts_df, min_count=10, min_samples_frac=0.1):
    """Apply trimmed mean of M-values normalization and convert to log2(CPM+1)."""
    np.random.seed(SEED)
    min_samples = max(1, int(counts_df.shape[1] * min_samples_frac))
    keep = (counts_df >= min_count).sum(axis=1) >= min_samples
    counts_df = counts_df.loc[keep].copy()

    lib_sizes = counts_df.sum(axis=0).astype(float)
    uq = lib_sizes.quantile(0.75)
    ref_col = (lib_sizes - uq).abs().idxmin()
    ref = counts_df[ref_col].astype(float)
    ref_ls = lib_sizes[ref_col]

    norm_factors = []
    for col in counts_df.columns:
        s = counts_df[col].astype(float)
        s_ls = lib_sizes[col]
        with np.errstate(divide="ignore", invalid="ignore"):
            M = np.log2(s / s_ls + 1e-9) - np.log2(ref / ref_ls + 1e-9)
            A = 0.5 * (np.log2(s / s_ls + 1e-9) + np.log2(ref / ref_ls + 1e-9))
        valid = np.isfinite(M) & np.isfinite(A) & (s > 0) & (ref > 0)
        if valid.sum() < 10:
            norm_factors.append(1.0)
            continue
        M_v, A_v = M[valid], A[valid]
        lo_m, hi_m = np.nanquantile(M_v, 0.30), np.nanquantile(M_v, 0.70)
        lo_a, hi_a = np.nanquantile(A_v, 0.05), np.nanquantile(A_v, 0.95)
        trim = (M_v >= lo_m) & (M_v <= hi_m) & (A_v >= lo_a) & (A_v <= hi_a)
        if trim.sum() < 5:
            norm_factors.append(1.0)
            continue
        w = (1.0 / (s_ls - s[valid][trim] + 1e-9)
             + 1.0 / (ref_ls - ref[valid][trim] + 1e-9))
        if w.sum() == 0:
            norm_factors.append(1.0)
            continue
        nf = float(2 ** np.average(M_v[trim], weights=w))
        norm_factors.append(nf)

    nf_arr = np.array(norm_factors, dtype=float)
    nf_arr = nf_arr / np.exp(np.nanmean(np.log(nf_arr + 1e-12)))
    nf_series = pd.Series(nf_arr, index=counts_df.columns)
    eff_lib = lib_sizes * nf_series
    cpm = counts_df.div(eff_lib, axis=1) * 1e6
    return np.log2(cpm + 1)


def load_tcga_counts(proj_dir, map_file):
    """Load STAR gene counts for one TCGA project. Filter to protein-coding genes
    using the Gencode v36 GTF annotation."""
    file_map = pd.read_csv(map_file)
    fname_col = "file_name" if "file_name" in file_map.columns else file_map.columns[1]
    barcode_col = "patient_id" if "patient_id" in file_map.columns else file_map.columns[2]
    counts_dir = proj_dir / "counts" if (proj_dir / "counts").exists() else proj_dir
    SKIP = ("#", "N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous", "__")

    def read_counts_file(fpath):
        col_index = 1
        opener = gzip.open if str(fpath).endswith(".gz") else open
        data_rows = []
        with opener(str(fpath), "rt", errors="replace") as fh:
            for line in fh:
                line = line.rstrip("\n\r")
                if not line:
                    continue
                parts = line.split("\t")
                gene_id = parts[0].lstrip("# ")
                if gene_id.lower() in ("gene_id", "geneid"):
                    try:
                        col_index = parts.index("unstranded")
                    except ValueError:
                        col_index = 1
                    continue
                if any(gene_id.startswith(p) for p in SKIP):
                    continue
                if not gene_id.startswith("ENSG"):
                    continue
                val = parts[col_index] if col_index < len(parts) else (parts[1] if len(parts) > 1 else "0")
                data_rows.append((gene_id.split(".")[0], val))
        if not data_rows:
            return None
        ids, vals = zip(*data_rows)
        series = pd.to_numeric(
            pd.Series(list(vals), index=list(ids)), errors="coerce"
        ).dropna()
        return series.groupby(series.index).max().astype(int) if not series.empty else None

    all_counts = {}
    for _, row in file_map.iterrows():
        fpath = counts_dir / str(row[fname_col])
        if not fpath.exists():
            continue
        cnt = read_counts_file(fpath)
        if cnt is None or cnt.empty:
            continue
        all_counts[str(row[barcode_col])[:12]] = cnt

    if not all_counts:
        raise RuntimeError(f"no counts loaded from {proj_dir}")
    counts_df = pd.DataFrame(all_counts).fillna(0).astype(int)

    # Map Ensembl to gene symbol and filter to protein-coding only
    gtf_path = proj_dir.parent / "gencode.v36.annotation.gtf.gz"
    if gtf_path.exists():
        rows = []
        with gzip.open(gtf_path, "rt") as f:
            for line in f:
                if line.startswith("#") or "\tgene\t" not in line:
                    continue
                info = {
                    k.strip(): v.strip('"')
                    for k, v in (
                        p.strip().split(" ", 1)
                        for p in line.split("\t")[8].split(";")
                        if " " in p.strip()
                    )
                }
                biotype = info.get("gene_biotype", "") or info.get("gene_type", "")
                rows.append({
                    "gene_id": info.get("gene_id", ""),
                    "gene_name": info.get("gene_name", ""),
                    "gene_type": biotype,
                })
        gene_info = pd.DataFrame(rows)
        gene_info["gene_id"] = gene_info["gene_id"].str.replace(r"\.\d+$", "", regex=True)
        gene_info = gene_info.set_index("gene_id")
        pc_genes = set(gene_info.loc[
            gene_info["gene_type"] == "protein_coding", "gene_name"
        ].dropna())
        name_map = gene_info["gene_name"].to_dict()
        counts_df.index = counts_df.index.map(lambda x: name_map.get(x, x))
        counts_df = counts_df.loc[counts_df.index.isin(pc_genes)]
        counts_df = counts_df.groupby(counts_df.index).max()
    return counts_df


In [ ]:
# Load TCGA-CDR
cdr_path_csv = RAW_DIR / "tcga_clinical" / "TCGA-CDR.csv"
cdr_path_xlsx = RAW_DIR / "tcga_clinical" / "TCGA-CDR.xlsx"
if cdr_path_csv.exists():
    cdr = pd.read_csv(cdr_path_csv)
elif cdr_path_xlsx.exists():
    cdr = pd.read_excel(cdr_path_xlsx)
else:
    raise FileNotFoundError("TCGA-CDR file not found")
print(f"CDR: {len(cdr)} patients, {cdr['type'].nunique()} cancer types")


In [ ]:
def clinical_from_cdr(cdr_sub, endpoint):
    """Extract clinical time/event for a given endpoint from CDR."""
    tcol, ecol = f"{endpoint}.time", endpoint
    if tcol not in cdr_sub.columns or ecol not in cdr_sub.columns:
        return pd.DataFrame(columns=["time", "event"])
    out = pd.DataFrame({
        "time": pd.to_numeric(cdr_sub[tcol], errors="coerce"),
        "event": pd.to_numeric(cdr_sub[ecol], errors="coerce"),
    }, index=cdr_sub.index).dropna()
    out["time"] = out["time"].astype(float)
    out["event"] = (out["event"].astype(float) > 0).astype(float)
    out.index = out.index.astype(str).str[:12]
    return out.loc[~out.index.duplicated(keep="first")]


def choose_tcga_endpoint(cdr_sub, proj, expr_ids):
    """Select the best clinical endpoint per the cancer-specific priority list.
    Falls back to remaining endpoints if the primary list yields insufficient data."""
    target_n = EXPECTED_N.get(proj)
    floor_n = FLOOR_N.get(proj, 100)
    need_n = max(int(0.60 * target_n) if target_n is not None else floor_n, floor_n)
    need_events = 20 if proj in ("TCGA-LUAD", "TCGA-KIRC") else 15

    def evaluate(ep):
        clin = clinical_from_cdr(cdr_sub, ep)
        if clin.empty:
            return None
        overlap = sorted(set(expr_ids) & set(clin.index))
        if not overlap:
            return None
        clin2 = clin.loc[overlap]
        return ep, clin2, len(clin2), int(clin2["event"].sum())

    primary = CANCER_ENDPOINT_PRIORITY.get(proj, DEFAULT_ENDPOINT_PRIORITY)
    for ep in primary:
        res = evaluate(ep)
        if res is None:
            continue
        ep_used, clin2, n, ev = res
        if n >= need_n and ev >= need_events:
            return ep_used, clin2

    fallback = [ep for ep in ["DSS", "OS", "PFI", "DFI"] if ep not in primary]
    best, best_score = None, (-1, -1)
    for ep in fallback:
        res = evaluate(ep)
        if res is None:
            continue
        ep_used, clin2, n, ev = res
        if (n, ev) > best_score:
            best_score = (n, ev)
            best = (ep_used, clin2)
    if best is None:
        raise ValueError(f"{proj}: no usable endpoint found")
    return best


In [ ]:
TCGA_PROJECTS = ["TCGA-BRCA", "TCGA-LUAD", "TCGA-KIRC", "TCGA-GBM"]

tcga_expr = {}
tcga_clinical = {}
tcga_endpoint_used = {}

for proj in TCGA_PROJECTS:
    print(f"\n{proj}")
    proj_dir = RAW_DIR / "tcga_rnaseq" / proj
    map_file = proj_dir / "patient_file_map.csv"
    if not map_file.exists():
        print(f"  patient_file_map.csv not found")
        continue

    counts = load_tcga_counts(proj_dir, map_file)
    log2cpm = tmm_log2cpm(counts)

    cdr_sub = cdr[cdr["type"] == proj.replace("TCGA-", "")].copy()
    if "bcr_patient_barcode" in cdr_sub.columns:
        cdr_sub = cdr_sub.set_index("bcr_patient_barcode")

    expr_ids = [str(c)[:12] for c in log2cpm.columns]
    log2cpm.columns = expr_ids
    log2cpm = log2cpm.loc[:, ~pd.Index(log2cpm.columns).duplicated(keep="first")]

    ep_used, clinical = choose_tcga_endpoint(cdr_sub, proj, expr_ids)
    tcga_endpoint_used[proj] = ep_used
    clinical = apply_time_units(clinical, proj, time_col_label=f"{ep_used}.time (days)")

    common = sorted(set(log2cpm.columns) & set(clinical.index))
    if len(common) < 50:
        raise ValueError(f"{proj}: only {len(common)} samples overlap")

    tcga_expr[proj] = log2cpm[common]
    tcga_clinical[proj] = clinical.loc[common]
    tcga_clinical[proj].attrs["endpoint_used"] = ep_used
    print(f"  endpoint {ep_used}: n={len(common)}, events={int(clinical.loc[common, 'event'].sum())}")
    validate_sample_size(proj, len(common))


## Batch correction

The three breast GEO cohorts are processed by ComBat to remove cross-platform
batch effects. GSE31210 is the only LUAD microarray cohort and is quantile
normalized independently. TCGA cohorts pass through the same downstream pipeline
without batch correction because they were already processed by a uniform STAR
+ TMM bioinformatic pipeline.


In [ ]:
try:
    from inmoose.pycombat import pycombat_norm
except ImportError:
    from combat.pycombat import pycombat as pycombat_norm

breast_ids = [c for c in ["GSE2034", "GSE2990", "GSE103746"] if c in geo_expr]
common_genes = sorted(set.intersection(*[set(geo_expr[c].index) for c in breast_ids]))
merged_unc = pd.concat([geo_expr[c].loc[common_genes] for c in breast_ids], axis=1)
batch_labels = []
for c in breast_ids:
    batch_labels.extend([c] * geo_expr[c].shape[1])
batch_series = pd.Series(batch_labels, index=merged_unc.columns)
batch_numeric = batch_series.map({c: i for i, c in enumerate(breast_ids)})

np.random.seed(SEED)
merged_corr = pycombat_norm(merged_unc, batch_numeric.values)
if not isinstance(merged_corr, pd.DataFrame):
    merged_corr = pd.DataFrame(
        merged_corr, index=merged_unc.index, columns=merged_unc.columns
    )

geo_expr_corrected = {}
geo_expr_uncorrected = {}
for c in breast_ids:
    mask = batch_series == c
    geo_expr_corrected[c] = merged_corr.loc[:, mask]
    geo_expr_uncorrected[c] = merged_unc.loc[:, mask]

if "GSE31210" in geo_expr:
    np.random.seed(SEED)
    arr = quantile_transform(
        geo_expr["GSE31210"].values, axis=1,
        output_distribution="normal", random_state=SEED,
    )
    geo_expr_corrected["GSE31210"] = pd.DataFrame(
        arr, index=geo_expr["GSE31210"].index, columns=geo_expr["GSE31210"].columns
    )
    geo_expr_uncorrected["GSE31210"] = geo_expr["GSE31210"].copy()

tcga_expr_corrected = dict(tcga_expr)
tcga_expr_uncorrected = dict(tcga_expr)

print("Batch correction complete")
for n in sorted(geo_expr_corrected):
    s = geo_expr_corrected[n]
    print(f"  {n}: {s.shape[0]} genes x {s.shape[1]} samples")


## Gene index cleaning


In [ ]:
def clean_expr_index(expr_df):
    """Remove invalid identifiers and collapse duplicates by mean."""
    df = expr_df.copy()
    df.index = pd.Index(list(map(str, df.index)))
    bad = (
        df.index.isin(["nan", "NaN", "None", "", "---"])
        | (df.index.str.strip() == "")
        | df.index.str.match(r"^\d+(\.\d+)?$")
    )
    df = df.loc[~bad]
    return df.groupby(df.index).mean()


for name in list(geo_expr_corrected.keys()):
    geo_expr_corrected[name] = clean_expr_index(geo_expr_corrected[name])
    geo_expr_uncorrected[name] = clean_expr_index(geo_expr_uncorrected[name])
for name in list(tcga_expr_corrected.keys()):
    tcga_expr_corrected[name] = clean_expr_index(tcga_expr_corrected[name])
    tcga_expr_uncorrected[name] = clean_expr_index(tcga_expr_uncorrected[name])


## ssGSEA pathway scoring

Score each sample against the combined MSigDB collection (Hallmark, KEGG legacy,
Reactome, GO Biological Process). The full collection contains roughly 6,200 to
6,650 gene sets after the size filter.


In [ ]:
def load_gmt(path):
    """Parse a GMT gene set file."""
    gs = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) > 2:
                gs[parts[0]] = parts[2:]
    return gs


GMT_DIR = RAW_DIR / "pathway_databases"
DB_CONFIG = [
    ("Hallmark", "h.all.gmt"),
    ("KEGG", "c2.cp.kegg_legacy.gmt"),
    ("Reactome", "c2.cp.reactome.gmt"),
    ("GO_BP", "c5.go.bp.gmt"),
]

pathway_dbs = {}
for db_name, fname in DB_CONFIG:
    p = GMT_DIR / fname
    if not p.exists():
        raise FileNotFoundError(f"missing GMT: {p}")
    pathway_dbs[db_name] = load_gmt(p)
    print(f"  {db_name}: {len(pathway_dbs[db_name])} gene sets")

combined_genesets = {}
for db in ["Hallmark", "KEGG", "Reactome", "GO_BP"]:
    combined_genesets.update(pathway_dbs[db])
print(f"  combined: {len(combined_genesets)} gene sets")


In [ ]:
def run_ssgsea(expr, gene_sets, name, min_size=10, max_size=500, chunk_size=800):
    """Run ssGSEA on an expression matrix. Pathways are filtered to those with
    at least min_size and at most max_size genes overlapping the expression index."""
    np.random.seed(SEED)
    expr_clean = expr.copy()
    expr_clean.index = pd.Index(list(map(str, expr_clean.index)))
    bad = (
        expr_clean.index.isin(["nan", "NaN", "None", "", "---"])
        | expr_clean.index.str.match(r"^\d+(\.\d+)?$")
    )
    expr_clean = (
        expr_clean.loc[~bad]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(how="all")
        .groupby(expr_clean.index[~bad]).mean()
    )
    expr_genes = set(expr_clean.index.str.upper())
    clean_gs = {
        k: [g for g in v if g.upper() in expr_genes]
        for k, v in gene_sets.items()
    }
    clean_gs = {k: v for k, v in clean_gs.items() if min_size <= len(v) <= max_size}
    if not clean_gs:
        raise ValueError(f"no gene sets survived overlap filter for {name}")

    keys = list(clean_gs.keys())
    n_chunks = int(math.ceil(len(keys) / chunk_size))
    chunks = [keys[i * chunk_size:(i + 1) * chunk_size] for i in range(n_chunks)]
    orig_cols = expr_clean.columns.tolist()
    pivots = []

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as tmp:
        tmp_path = tmp.name
        expr_clean.to_csv(tmp_path, sep="\t")

    try:
        for ck in chunks:
            sub_gs = {k: clean_gs[k] for k in ck}
            res = gp.ssgsea(
                data=tmp_path, gene_sets=sub_gs, outdir=None,
                min_size=min_size, max_size=max_size,
                no_plot=True, verbose=False,
                permutation_num=0, threads=1,
            )
            df = res.res2d if hasattr(res, "res2d") else (
                res.results if hasattr(res, "results") else res
            )
            term_col = next(
                (c for c in df.columns if c.lower() in ("term", "pathway", "geneset")),
                None,
            )
            sample_col = next(
                (c for c in df.columns if c.lower() in ("name", "sample", "cell line")),
                None,
            )
            nes_col = next(
                (c for c in df.columns if c.lower() in ("nes", "es", "norm es")),
                None,
            )
            if term_col and sample_col and nes_col:
                pivot = df.pivot_table(
                    index=term_col, columns=sample_col,
                    values=nes_col, aggfunc="mean",
                )
            else:
                pivot = df.iloc[:, :3].pivot_table(
                    index=df.columns[0], columns=df.columns[1],
                    values=df.columns[2], aggfunc="mean",
                )
            pivot = pivot.reindex(columns=orig_cols)
            pivots.append(pivot)
        out = pd.concat(pivots, axis=0)
        out = out[~out.index.duplicated(keep="first")]
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
    return out


In [ ]:
# Run ssGSEA on batch-corrected expression
all_corrected = {**geo_expr_corrected, **tcga_expr_corrected}
ssgsea_corrected = {}
for name in all_corrected:
    print(f"  ssGSEA: {name}")
    np.random.seed(SEED)
    ssgsea_corrected[name] = run_ssgsea(all_corrected[name], combined_genesets, name)
    print(f"    {ssgsea_corrected[name].shape[0]} pathways x {ssgsea_corrected[name].shape[1]} samples")

# Run ssGSEA on uncorrected expression for batch sensitivity check
all_uncorrected = {**geo_expr_uncorrected, **tcga_expr_corrected}
ssgsea_raw = {}
for name in all_uncorrected:
    print(f"  ssGSEA uncorrected: {name}")
    np.random.seed(SEED)
    ssgsea_raw[name] = run_ssgsea(all_uncorrected[name], combined_genesets, name)


## Save outputs


In [ ]:
def save_pkl(obj, name):
    p = INTER_DIR / f"{name}.pkl"
    with open(p, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  {name}.pkl ({p.stat().st_size / 1e6:.1f} MB)")


save_pkl(geo_clinical, "geo_clinical")
save_pkl(tcga_clinical, "tcga_clinical")
save_pkl(tcga_endpoint_used, "tcga_endpoint_used")
save_pkl(ssgsea_corrected, "ssgsea_corrected")
save_pkl(ssgsea_raw, "ssgsea_raw")
save_pkl(geo_expr_corrected, "geo_expr_corrected")
save_pkl(tcga_expr_corrected, "tcga_expr_corrected")
save_pkl(combined_genesets, "combined_genesets")

print("\nNext: 02_cosmos_feature_selection.ipynb")
